In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [9]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor(),
])

caltech101_dataset = datasets.Caltech101(root='../caltech_data', download=False, transform=transform)

test_loader = DataLoader(caltech101_dataset, batch_size=1, shuffle=False)

NUM_CLASSES = 101

In [10]:
def load_resnet34(weights_path, num_classes=NUM_CLASSES, device=device):

    model = models.resnet34(weights=None)
    
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    
    checkpoint = torch.load(weights_path, map_location=device)
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])
    elif isinstance(checkpoint, dict):
        model.load_state_dict(checkpoint)
    else:
        model = checkpoint

    model = model.to(device)
    model.eval()
    return model


def load_mobilenet_v2(weights_path, num_classes=NUM_CLASSES, device=device):
    model = models.mobilenet_v2(weights=None)
    
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    
    checkpoint = torch.load(weights_path, map_location=device)
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])
    elif isinstance(checkpoint, dict):
        model.load_state_dict(checkpoint)
    else:
        model = checkpoint

    model = model.to(device)
    model.eval()
    return model

In [11]:
resnet34_path = "../Phase1/resnet34_caltech101.pth"
mobilenet_path = "../Phase1/mobilenetv2_caltech101.pth"


resnet34_model = load_resnet34(resnet34_path)
print("ResNet-34 successfully loaded")

mobilenet_model = load_mobilenet_v2(mobilenet_path)
print("MobileNetV2 successfully loaded")

ResNet-34 successfully loaded
MobileNetV2 successfully loaded


# FGSM Attack Code

In [12]:
def fgsm_attack(image, epsilon, data_grad):
    # Get the direction of the gradient (either +1 or -1 for every pixel)
    sign_data_grad = data_grad.sign()
    
    # Multiply by epsilon (attack strength) and add to the original image
    # Adds "noise" to the image
    perturbed_image = image + epsilon * sign_data_grad
    
    return perturbed_image

# Testing the attack

In [13]:
def test_fgsm(model, device, test_loader, epsilon, max_samples=50):
    correct = 0
    adv_examples = []
    processed_count = 0

    for data, target in test_loader:
        # Stop once we hit our sample limit
        if processed_count >= max_samples:
            break
            
        data, target = data.to(device), target.to(device)
        data.requires_grad = True

        output = model(data)
        init_pred = output.max(1, keepdim=True)[1]

        # Skip false predictions
        if init_pred.item() != target.item():
            continue

        loss = F.cross_entropy(output, target)
        model.zero_grad()
        loss.backward()
        data_grad = data.grad.data

        perturbed_data = fgsm_attack(data, epsilon, data_grad)
        output = model(perturbed_data)
        final_pred = output.max(1, keepdim=True)[1]

        if final_pred.item() == target.item():
            correct += 1
            if (epsilon == 0) and (len(adv_examples) < 5):
                adv_examples.append((init_pred.item(), final_pred.item(), perturbed_data.squeeze().detach().cpu()))
        else:
            if len(adv_examples) < 5:
                adv_examples.append((init_pred.item(), final_pred.item(), perturbed_data.squeeze().detach().cpu()))

        processed_count += 1

    final_acc = correct / float(processed_count)
    print(f"Epsilon: {epsilon:.2f}\tAccuracy = {correct}/{processed_count} ({final_acc * 100:.2f}%)")
    return final_acc, adv_examples

In [14]:
# Test across a range of epsilons
epsilons = [0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]

print("--- Testing ResNet-34 ---")
resnet_accs = [test_fgsm(resnet34_model, device, test_loader, eps)[0] for eps in epsilons]

print("\n--- Testing MobileNetV2 ---")
mobilenet_accs = [test_fgsm(mobilenet_model, device, test_loader, eps)[0] for eps in epsilons]

--- Testing ResNet-34 ---
Epsilon: 0.00	Accuracy = 50/50 (100.00%)
Epsilon: 0.05	Accuracy = 40/50 (80.00%)
Epsilon: 0.10	Accuracy = 43/50 (86.00%)
Epsilon: 0.15	Accuracy = 43/50 (86.00%)
Epsilon: 0.20	Accuracy = 43/50 (86.00%)
Epsilon: 0.25	Accuracy = 45/50 (90.00%)
Epsilon: 0.30	Accuracy = 48/50 (96.00%)

--- Testing MobileNetV2 ---
Epsilon: 0.00	Accuracy = 50/50 (100.00%)
Epsilon: 0.05	Accuracy = 48/50 (96.00%)
Epsilon: 0.10	Accuracy = 33/50 (66.00%)
Epsilon: 0.15	Accuracy = 8/50 (16.00%)
Epsilon: 0.20	Accuracy = 1/50 (2.00%)
Epsilon: 0.25	Accuracy = 0/50 (0.00%)
Epsilon: 0.30	Accuracy = 0/50 (0.00%)
